# 1. Setup

Compare single, multi and reviewed disease-attribute prompts using one unchanged frozen checkpoint. Supply 3–5 verified symptom phrases per included training disease in ATTRIBUTES before evaluation. Diseases without entries use the ordinary multi-template bank. With an empty map, run single and multi only.

In [ ]:
import gc, json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, collect_scores, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
RUN_FOLDER = None  # Completed full 3Z run, or depth_0 folder from the ablation.
cfg.NUM_WORKERS = 0
cfg.set_seed(cfg.SEED)
MODEL_NAME = 'approach3z_v3_prompt_ablation'
ATTRIBUTES = {}

# 2. Data

Use training-derived names and pair order.

In [ ]:
pairs = get_pairs()
concept_names = get_crop_and_disease_names()

# 3. Model

Only the pair prompt bank varies. Crop and disease banks and scorer are shared across arms.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self, depth):
        if depth != 0:
            raise ValueError('3Z requires both DINOv2 towers frozen (depth 0).')
        self.requires_grad_(False).eval()

    @torch.no_grad()
    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

class TextOnly:
    """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
    def __init__(self,config,device,allow_download=False,revision=None):
        from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
        path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
            ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
        self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
        self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
        if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
        self.dimension=self.model.config.projection_dim;self.device=device
        if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
        weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,
            class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
    @torch.no_grad()
    def encode(self,prompts,size):
        self.assert_frozen();result=[]
        for start in range(0,len(prompts),size):
            tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
            result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
        return torch.cat(result)
    def assert_frozen(self):
        if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
            raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')

def text_banks(text,pairs,names,config,ensemble=False):
    crops,diseases=names
    def clean(x):return x.replace('_',' ')
    pair_groups=[]
    for c,d in pairs.tolist():
        if clean(diseases[d]).strip().casefold()=='healthy':
            templates=config['healthy_templates'] if ensemble else config['healthy_templates'][:1]
        else:templates=config['pair_templates'] if ensemble else config['pair_templates'][:1]
        pair_groups.append([t.format(crop=clean(crops[c]),disease=clean(diseases[d])) for t in templates])
    flat=[p for group in pair_groups for p in group];encoded=text.encode(flat,config['text_batch_size'])
    bank=[];offset=0
    for group in pair_groups:
        bank.append(F.normalize(encoded[offset:offset+len(group)].mean(0),dim=0));offset+=len(group)
    crop_prompts=[f'a photo of a {clean(crops[i])} plant' for i in range(cfg.NUM_CROPS)]
    disease_prompts=[('a photo of a healthy leaf' if clean(diseases[i]).casefold()=='healthy' else f'a leaf showing {clean(diseases[i])}') for i in range(cfg.NUM_DISEASES)]
    return [torch.stack(bank),text.encode(crop_prompts,config['text_batch_size']),text.encode(disease_prompts,config['text_batch_size'])],dict(pair=pair_groups,crop=crop_prompts,disease=disease_prompts)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))

def zscore(x):
    return (x-x.mean(1, keepdim=True)) / x.std(1, unbiased=False, keepdim=True).clamp_min(1e-6)

def factorized(parts, pairs, weights):
    pair, crop, disease = parts
    if len(weights) != 3 or min(weights) < 0 or abs(sum(weights)-1) > 1e-6:
        raise ValueError('Three nonnegative scorer weights summing to one required.')
    pairs = pairs.to(pair.device)
    return weights[0]*zscore(pair)+weights[1]*zscore(crop)[:, pairs[:,0]]+weights[2]*zscore(disease)[:, pairs[:,1]]


RECIPE = dict(pair=1., primitive=1.)


class Model(nn.Module):
    def __init__(self, backbone, banks):
        super().__init__()
        self.backbone = backbone
        width, hidden = backbone.width, HP['hidden_dim']
        dim = banks[0].shape[1]
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)

        self.pair_projection = projection(width, hidden, dim)
        self.log_temperature = nn.Parameter(torch.tensor(math.log(1 / .07)))
        for index, bank in enumerate(banks):
            self.register_buffer(f'text{index}', bank)

    def install_text(self, banks):
        for index, bank in enumerate(banks):
            setattr(self, f'text{index}', bank)

    def forward(self, pixels):
        crop_features, disease_features = self.backbone(pixels)
        crop = F.normalize(self.crop_projection(crop_features), dim=1)
        disease = F.normalize(self.disease_projection(disease_features), dim=1)
        pair = F.normalize(self.pair_projection((crop_features + disease_features) / 2), dim=1)
        scale = self.log_temperature.clamp(0., math.log(100.)).exp()
        return scale * pair @ self.text0.T, scale * crop @ self.text1.T, scale * disease @ self.text2.T


def make_model(pairs, names, depth, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(depth)
    text, banks = None, None

    text = TextOnly(HP, cfg.DEVICE, allow_download=True, revision=(metadata.get('text') or {}).get('revision'))
    banks, _ = text_banks(text, pairs, names, HP)

    torch.manual_seed(cfg.SEED)
    model = Model(backbone, banks).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, text, dict(dino=backbone.metadata, text=text.metadata if text else None)


def make_predict(model, candidate_pairs):
    @torch.no_grad()
    def predict(images):
        model.eval()
        return factorized(model(images.to(cfg.DEVICE)), candidate_pairs, HP['score_weights'])
    return predict
def load_frozen_run(folder):
    folder = Path(folder)
    selection = json.loads((folder / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(folder / selection['checkpoint'], map_location='cpu', weights_only=True)
    if (selection.get('model_name') not in ('approach3z_dino_frozen_text_alignment', 'approach3z_depth_ablation',
                                                 'approach3z_v1_dino_frozen_text_alignment', 'approach3z_v2_depth_ablation')
            or selection.get('family') != 'B' or selection.get('debug') is not False or selection.get('depth') != 0
            or selection.get('recipe') != RECIPE):
        raise ValueError('Expected a completed non-DEBUG 3Z depth-zero run.')
    fields = ('model_name', 'family', 'debug', 'depth', 'hp', 'recipe', 'best_epoch', 'best_val')
    if (any(checkpoint.get(key) != selection.get(key) for key in fields)
            or not torch.equal(checkpoint['pairs'], pairs) or selection['development_pairs'] != pairs.tolist()):
        raise ValueError('Checkpoint and frozen selection differ.')
    hp = checkpoint['hp']
    if (hp['backbone'], hp['norm'], hp['seed']) != (MODEL, 'imagenet', cfg.SEED):
        raise ValueError('Checkpoint settings differ from the shared protocol.')
    train, val, _, _ = get_loaders(hp['batch_size'], norm=hp['norm'], debug=False)
    if (selection['train_images'] != train.dataset.frame.image.tolist()
            or selection['validation_images'] != val.dataset.frame.image.tolist()):
        raise ValueError('Shared train/validation membership changed.')
    return checkpoint, val


def restore(checkpoint):
    model, text, metadata = make_model(pairs, concept_names, 0, checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public model revision changed.')
    model.load_state_dict(checkpoint['state_dict'])
    return model.eval().requires_grad_(False), text

def attribute_bank(text, candidate_pairs, attributes):
    crops, diseases = concept_names
    groups = []
    for crop_id, disease_id in candidate_pairs.tolist():
        crop, disease = crops[crop_id].replace('_', ' '), diseases[disease_id].replace('_', ' ')
        if disease.strip().casefold() == 'healthy':
            prompts = [template.format(crop=crop) for template in HP['healthy_templates']]
        else:
            prompts = [template.format(crop=crop, disease=disease) for template in HP['pair_templates']]
            prompts.extend(f'a photo of a {crop} leaf with {disease}, showing {cue}'
                           for cue in attributes.get(disease, ()))
        groups.append(prompts)
    encoded = text.encode([prompt for group in groups for prompt in group], HP['text_batch_size'])
    return torch.stack([F.normalize(chunk.mean(0), dim=0)
                        for chunk in encoded.split([len(group) for group in groups])])

# 4. Prepare

Validate the checkpoint and freeze the attribute map before final access. No retraining.

In [ ]:
ready = False
if RUN_FOLDER is None:
    print('Set RUN_FOLDER to a completed frozen run.')
else:
    checkpoint, val_loader = load_frozen_run(RUN_FOLDER)
    HP = checkpoint['hp']
    model, text = restore(checkpoint)
    del checkpoint

    known = {label.replace('_', ' ') for label in concept_names[1].values()}
    frozen_attributes = {}
    for disease, cues in ATTRIBUTES.items():
        if disease not in known or disease.strip().casefold() == 'healthy':
            raise ValueError(f'Use a training-derived disease name for attributes: {disease}')
        if not isinstance(cues, (list, tuple)) or not 3 <= len(cues) <= 5 or any(
                not isinstance(cue, str) or not cue.strip() for cue in cues):
            raise ValueError('Supply 3–5 reviewed symptom phrases per included disease.')
        frozen_attributes[disease] = tuple(cues)
    arms = ('single', 'multi', 'attributes') if frozen_attributes else ('single', 'multi')
    ready = True

# 5. Evaluation

Use identical images and candidates for each prompt mode.

In [ ]:
RUN_FINAL_EVALUATION = False
reports = []
if RUN_FINAL_EVALUATION:
    if not ready:
        raise ValueError('Prepare the frozen checkpoint first.')
    final_pairs = get_pairs(final_evaluation=True)
    seen, unseen = get_final_test_loaders(HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    RESULTS = cfg.RESULTS_DIR / (MODEL_NAME + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    RESULTS.mkdir(parents=True, exist_ok=False)

    previous = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        for arm in arms:
            banks, _ = text_banks(text, final_pairs, concept_names, HP, ensemble=arm != 'single')
            if arm == 'attributes':
                banks[0] = attribute_bank(text, final_pairs, frozen_attributes)
            model.install_text(banks)
            predict = make_predict(model, final_pairs)
            name = f'{MODEL_NAME}_{arm}'
            settings = dict(prompt_mode=arm, attributes=frozen_attributes if arm == 'attributes' else {},
                            pair_templates=HP['pair_templates'], healthy_templates=HP['healthy_templates'])
            result = evaluate(predict, seen, unseen, name, settings, robust=True)
            scores, labels = collect_scores(predict, unseen)
            crop_acc, disease_acc = concept_accuracy(scores, labels, final_evaluation=True)
            result.update(prompt_mode=arm, unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc)
            (RESULTS / f'{name}.json').write_text(json.dumps(result, indent=2), encoding='utf8')
            reports.append(result)
    finally:
        cfg.RESULTS_DIR = previous

# 6. Save

Export the comparison.

In [ ]:
if reports:
    table = pd.DataFrame(reports).drop(columns=['hp', 'robustness'])
    table.to_csv(RESULTS / 'prompt_comparison.csv', index=False)
    display(table)